# July 1 dippers and long-term variables: multi-survey photometry

This notebook reads the live July 1 review database in read-only mode and combines two reviewed cohorts: candidates marked `dipper` with cached multi-epoch NEOWISE photometry, and candidates marked `ltv` with resolved cached AllWISE multi-epoch photometry. It plots every cached time-domain survey through MALCA's shared review plotting stack. For every cohort member with a locally displayable spectrum, it also renders the preferred cached APOGEE, DESI, or LAMOST spectrum directly beneath that candidate's photometry. No catalog queries or photometry fetches are performed.

Each figure uses one shared JD panel with independent overlaid magnitude axes for ASAS-SN, WISE/NEOWISE, ZTF, and PS1. ASAS-SN uses the base left axis, WISE uses the base right axis, and available ZTF/PS1 data receive outward left/right axes; every survey therefore shares the same time coordinate without sharing an inappropriate magnitude scale. Other cached surveys remain overplotted on the base optical axis. TESS flux is converted to relative magnitude by the shared MALCA trace builder, split into every contiguous observing window, and drawn in chronological small multiples beneath the main panel. A time-aligned coverage strip and matching window labels preserve each TESS panel's location on the main JD axis. Figures use MALCA's established two-column publication width, typography, axis styling, and coordinate header boxes. Spectrum panels use the existing MALCA publication spectrum renderer, including continuum and normalized-residual views.

## Setup

In [1]:
from __future__ import annotations

from contextlib import closing
from dataclasses import replace
from pathlib import Path
import sqlite3
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image as NotebookImage, display
from matplotlib.ticker import FormatStrFormatter, MultipleLocator

warnings.filterwarnings('ignore', message='IProgress not found.*')
warnings.filterwarnings('ignore', message='You passed a edgecolor/edgecolors.*unfilled marker.*')


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in (start, *start.parents):
        if (path / 'pyproject.toml').exists() and (path / 'malca').is_dir():
            return path
    raise FileNotFoundError('Could not locate the MALCA repository root.')


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from malca.plotting.lightcurve_publication import (
    FIG_TWO_COL_WIDTH,
    PUBLICATION_STYLE,
)
from malca.enrich.spectrum_fetch import FetchStatus, fetch_spectrum
from malca.review.lightcurve_assembly import (
    ReviewPlotRequest,
    assemble_review_lightcurve_plot,
)
from malca.review.lightcurve_pdf import (
    _axis_label_for_offset,
    _draw_header_boxes,
    _mpl_marker,
    _style_lightcurve_axis,
)
from malca.review.lightcurve_sources import (
    EXTERNAL_SOURCE_ORDER,
    discover_external_lcs,
    external_source_label,
)
from malca.review.spectrum_plot import render_spectrum_png
from malca.review.store import get_candidate_payload

## Configuration

In [2]:
RUN_DIR = REPO_ROOT / 'output' / 'runs' / 'dat3-full-extended_2026-07-01-v4'
REVIEW_DB = RUN_DIR / 'review' / 'review.db'
SPECTRA_LONG = RUN_DIR / 'results' / 'spectra_enrichment' / 'spectra_long.parquet'
SPECTRUM_CACHE_DIR = RUN_DIR / 'review' / 'spectrum_cache'

FILTER_BAD_CAMERAS = True
SHOW_EVENT_MARKERS = False
SHOW_TESS_PANELS = True
FIGSIZE = (2.0 * FIG_TWO_COL_WIDTH, 5.75)
TESS_GAP_DAYS = 5.0  # gaps larger than this split TESS observing windows
TESS_PANELS_PER_ROW = 1
TESS_ROW_HEIGHT = 1.55
OUTER_AXIS_OFFSET_POINTS = 52.0
MAIN_MARGIN_WITH_OUTER_AXIS = 0.19
MAIN_MARGIN_WITHOUT_OUTER_AXIS = 0.11
CANDIDATE_INDEX = 0
PLOT_ALL = True

# Reviewed dippers and LTVs with locally renderable spectrum backends.
SPECTRUM_SOURCE_PRIORITY = {
    'stv_111669291649': ('apogee_dr16',),
    'stv_171799666012': ('lamost_dr7',),
    'stv_214748792627': ('lamost_dr7',),
    'stv_223338997633': ('desi_dr1',),
    'stv_223339033515': ('lamost_dr7',),
    'stv_249108767924': ('lamost_dr7',),
    'stv_343598086600': ('apogee_dr16', 'lamost_dr7'),
    'stv_34360872825': ('lamost_dr7',),
    'stv_60130665647': ('lamost_dr7',),
    'stv_77309467421': ('lamost_dr7',),
    'stv_94489786439': ('apogee_dr16', 'lamost_dr7'),
}
SPECTRUM_SOURCE_LABELS = {
    'apogee_dr16': 'APOGEE DR16',
    'desi_dr1': 'DESI DR1',
    'lamost_dr7': 'LAMOST DR7',
}
# Parquet coerced this 64-bit DESI ID to float; retain its exact integer here.
SPECTRUM_ROW_OVERRIDES = {
    ('stv_223338997633', 'desi_dr1'): {
        'TargetID': 2305843028272617471,
    },
}

TIME_DOMAIN_SOURCES = tuple(
    source
    for source in EXTERNAL_SOURCE_ORDER
    if source not in {'asassn', 'neowise_w1', 'neowise_w2', 'neowise_color'}
)

if not REVIEW_DB.exists():
    raise FileNotFoundError(REVIEW_DB)
if not SPECTRA_LONG.exists():
    raise FileNotFoundError(SPECTRA_LONG)

spectrum_rows = pd.read_parquet(SPECTRA_LONG)
spectrum_rows = spectrum_rows.loc[
    spectrum_rows['candidate_id'].astype(str).isin(SPECTRUM_SOURCE_PRIORITY)
].copy()

print(f'Run directory: {RUN_DIR}')
print(f'Review DB:     {REVIEW_DB}')
print(f'Spectrum rows: {len(spectrum_rows)} for {spectrum_rows["candidate_id"].nunique()} dippers')

Run directory: /Users/calder/code/malca/output/runs/dat3-full-extended_2026-07-01-v4
Review DB:     /Users/calder/code/malca/output/runs/dat3-full-extended_2026-07-01-v4/review/review.db
Spectrum rows: 50 for 11 dippers


## Load the live dipper + NEOWISE and LTV + AllWISE cohorts

The SQLite URI uses `mode=ro`, so this notebook sees committed WAL content without modifying the review database. The shared external-light-curve resolver checks the run manifest/cache; it does not fetch missing data.

In [3]:
db_uri = f'file:{REVIEW_DB.resolve()}?mode=ro'
with closing(sqlite3.connect(db_uri, uri=True, timeout=30.0)) as conn:
    cohort = pd.read_sql_query(
        """
        SELECT
            r.candidate_id,
            r.interest_score,
            r.event_class,
            r.updated_at,
            CASE
                WHEN lower(trim(r.event_class)) = 'dipper'
                    THEN 'Dipper + NEOWISE'
                WHEN lower(trim(r.event_class)) = 'ltv'
                    THEN 'LTV + AllWISE'
            END AS cohort_group,
            c.neowise_n_epochs,
            c.neowise_w1_range,
            c.neowise_w2_range
        FROM reviews AS r
        JOIN candidates AS c USING (candidate_id)
        WHERE (
            lower(trim(r.event_class)) = 'dipper'
            AND COALESCE(c.neowise_n_epochs, 0) > 0
        )
        OR lower(trim(r.event_class)) = 'ltv'
        ORDER BY cohort_group, r.updated_at, r.candidate_id
        """,
        conn,
    )
    payload_by_id = {
        candidate_id: get_candidate_payload(conn, candidate_id)
        for candidate_id in cohort['candidate_id'].astype(str)
    }

external_paths_by_id: dict[str, dict[str, Path]] = {}
for candidate_id, payload in payload_by_id.items():
    external_paths_by_id[candidate_id] = discover_external_lcs(
        candidate_id,
        payload,
        RUN_DIR,
        list(TIME_DOMAIN_SOURCES),
    )

is_dipper = cohort['event_class'].astype(str).str.strip().str.lower().eq('dipper')
has_allwise_mep = cohort['candidate_id'].astype(str).map(
    lambda candidate_id: 'allwise_mep' in external_paths_by_id.get(candidate_id, {})
)
cohort = cohort.loc[is_dipper | has_allwise_mep].reset_index(drop=True)
selected_ids = set(cohort['candidate_id'].astype(str))
payload_by_id = {
    candidate_id: payload
    for candidate_id, payload in payload_by_id.items()
    if candidate_id in selected_ids
}
external_paths_by_id = {
    candidate_id: paths
    for candidate_id, paths in external_paths_by_id.items()
    if candidate_id in selected_ids
}

cohort['time_domain_sources'] = cohort['candidate_id'].astype(str).map(
    lambda candidate_id: ', '.join(
        ['ASAS-SN']
        + [
            external_source_label(source)
            for source in TIME_DOMAIN_SOURCES
            if source in external_paths_by_id.get(candidate_id, {})
        ]
    )
)

source_summary = pd.DataFrame(
    [
        {'source': 'asassn', 'label': 'ASAS-SN', 'candidates': len(cohort)},
        *[
            {
                'source': source,
                'label': external_source_label(source),
                'candidates': sum(
                    source in external_paths_by_id.get(candidate_id, {})
                    for candidate_id in cohort['candidate_id'].astype(str)
                ),
            }
            for source in TIME_DOMAIN_SOURCES
        ],
    ]
)
source_summary = source_summary.loc[source_summary['candidates'] > 0].reset_index(drop=True)

dipper_ids = set(
    cohort.loc[cohort['event_class'].astype(str).str.lower().eq('dipper'), 'candidate_id'].astype(str)
)
ltv_ids = set(
    cohort.loc[cohort['event_class'].astype(str).str.lower().eq('ltv'), 'candidate_id'].astype(str)
)
resolved_dipper_neowise = sum(
    'neowise' in external_paths_by_id.get(candidate_id, {})
    for candidate_id in dipper_ids
)

print(f'Marked dippers with NEOWISE epochs:    {len(dipper_ids)}')
print(f'Marked LTVs with cached AllWISE MEP:    {len(ltv_ids)}')
print(f'Combined deduplicated cohort:           {len(cohort)}')
print(f'Cached NEOWISE files resolved:          {sum("neowise" in paths for paths in external_paths_by_id.values())}')
print(f'Cached AllWISE MEP files resolved:       {sum("allwise_mep" in paths for paths in external_paths_by_id.values())}')
print(f'Cached TESS files resolved:              {sum("tess" in paths for paths in external_paths_by_id.values())}')
if resolved_dipper_neowise != len(dipper_ids):
    print('Warning: some DB-positive dippers do not currently resolve to cached NEOWISE files.')
display(source_summary)
display(
    cohort[[
        'candidate_id',
        'event_class',
        'cohort_group',
        'interest_score',
        'neowise_n_epochs',
        'neowise_w1_range',
        'neowise_w2_range',
        'time_domain_sources',
    ]]
)

DatabaseError: Execution failed on sql '
        SELECT
            r.candidate_id,
            r.interest_score,
            r.event_class,
            r.updated_at,
            CASE
                WHEN lower(trim(r.event_class)) = 'dipper'
                    THEN 'Dipper + NEOWISE'
                WHEN lower(trim(r.event_class)) = 'ltv'
                    THEN 'LTV + AllWISE'
            END AS cohort_group,
            c.neowise_n_epochs,
            c.neowise_w1_range,
            c.neowise_w2_range
        FROM reviews AS r
        JOIN candidates AS c USING (candidate_id)
        WHERE (
            lower(trim(r.event_class)) = 'dipper'
            AND COALESCE(c.neowise_n_epochs, 0) > 0
        )
        OR lower(trim(r.event_class)) = 'ltv'
        ORDER BY cohort_group, r.updated_at, r.candidate_id
        ': no such column: r.interest_score

## Shared MALCA plotting helper

This helper delegates ASAS-SN loading/cleaning, external-survey normalization, time-axis conversion, flux-to-relative-magnitude conversion, colors, markers, and trace construction to the same modules used by `malca review`. Notebook code only composes the publication-style Matplotlib panels, splits TESS data at genuine observing gaps, maps every resulting window back to the main JD timeline, and selects a preferred cached spectrum. Spectrum analysis and rendering remain in MALCA's shared review modules.

In [ ]:
def build_time_domain_spec(candidate_id: str):
    candidate_id = str(candidate_id)
    if candidate_id not in payload_by_id:
        raise KeyError(f'Candidate is not in the current cohort: {candidate_id}')

    external_lcs = external_paths_by_id.get(candidate_id, {})
    source_names = [source for source in TIME_DOMAIN_SOURCES if source in external_lcs]
    request = ReviewPlotRequest.from_kwargs(
        payload_by_id[candidate_id],
        plot_dir=RUN_DIR,
        selected_cameras=None,
        filter_bad_cameras=FILTER_BAD_CAMERAS,
        show_baseline=False,
        show_event_markers=SHOW_EVENT_MARKERS,
        show_residuals=False,
        show_phase_fold=False,
        show_raw_mag=True,
        show_diagnostics=False,
        confidence_colors=False,
        run_params=None,
        yaxis_mode='mag',
        external_lcs=external_lcs,
        external_source_view=['asassn', *source_names],
        external_panel_mode='overlay',
        selected_bands=None,
        native_color_mode='band',
        candidate_id=candidate_id,
        discover_external=False,
    )
    spec = assemble_review_lightcurve_plot(request)
    if spec.status != 'ok':
        raise RuntimeError(f'{candidate_id}: {spec.status}: {spec.status_message}')
    return spec, source_names


def split_tess_windows(spec):
    tess_traces = [
        trace
        for trace in spec.traces
        if trace.panel_id == 'raw' and str(trace.label or '').startswith('TESS')
    ]
    windows = []
    for trace in tess_traces:
        x = np.asarray(trace.x, dtype=float)
        y = np.asarray(trace.y, dtype=float)
        finite = np.flatnonzero(np.isfinite(x) & np.isfinite(y))
        if finite.size < 20:
            continue

        ordered = finite[np.argsort(x[finite])]
        split_at = np.flatnonzero(np.diff(x[ordered]) > TESS_GAP_DAYS) + 1
        chunks = [chunk for chunk in np.split(ordered, split_at) if chunk.size >= 20]
        for chunk in chunks:
            window_yerr = None
            if trace.yerr is not None:
                window_yerr = np.asarray(trace.yerr, dtype=float)[chunk]
            windows.append(
                replace(
                    trace,
                    x=x[chunk],
                    y=y[chunk],
                    yerr=window_yerr,
                    label='TESS window',
                    showlegend=False,
                    customdata=None,
                    marker_size=2.4,
                )
            )

    windows.sort(key=lambda trace: float(np.nanmin(np.asarray(trace.x, dtype=float))))
    return windows


def trace_group(trace):
    label = str(trace.label or '')
    if label.startswith('TESS'):
        return 'tess'
    if label.startswith('ZTF '):
        return 'ztf'
    if label.startswith('PS1 '):
        return 'ps1'
    if label.startswith(('NEOWISE ', 'AllWISE ')):
        return 'infrared'
    return 'optical'


def plot_standard_trace(ax, trace, *, alpha: float, marker_size: float, label: str):
    x = np.asarray(trace.x, dtype=float)
    y = np.asarray(trace.y, dtype=float)
    finite = np.isfinite(x) & np.isfinite(y)
    if not finite.any():
        return None

    x = x[finite]
    y = y[finite]
    yerr = None
    if trace.yerr is not None:
        err = np.asarray(trace.yerr, dtype=float)[finite]
        valid_err = np.isfinite(err) & (err >= 0)
        if valid_err.any():
            yerr = np.where(valid_err, err, 0.0)

    marker = _mpl_marker(trace.marker)
    color = trace.color or '0.25'
    size = float(marker_size)
    if x.size >= 5000:
        size = min(size, 1.15)
    elif x.size >= 1000:
        size = min(size, 1.55)

    kwargs = {
        'fmt': marker,
        'linestyle': 'none',
        'markersize': size,
        'color': color,
        'ecolor': color,
        'elinewidth': 0.35,
        'capsize': 0.0,
        'alpha': float(alpha),
        'label': label,
        'zorder': 4,
    }
    if marker in {'x', '+', '1', '2', '3', '4', '|', '_'}:
        kwargs['markeredgecolor'] = color
        kwargs['markeredgewidth'] = 0.55
    else:
        kwargs['markerfacecolor'] = color
        kwargs['markeredgecolor'] = 'white'
        kwargs['markeredgewidth'] = 0.25
    return ax.errorbar(x, y, yerr=yerr, **kwargs)


def robust_magnitude_limits(traces):
    values = [
        np.asarray(trace.y, dtype=float)[np.isfinite(np.asarray(trace.y, dtype=float))]
        for trace in traces
    ]
    values = [value for value in values if value.size]
    if not values:
        return None
    finite = np.concatenate(values)
    lo, hi = np.nanpercentile(finite, [0.2, 99.8]) if finite.size >= 50 else (np.nanmin(finite), np.nanmax(finite))
    span = max(0.03, float(hi - lo))
    pad = max(0.02, 0.06 * span)
    return float(hi + pad), float(lo - pad)


def configure_overlay_magnitude_axis(ax, *, side: str, outward: float, label: str, limits):
    _style_lightcurve_axis(ax)
    ax.grid(False, which='both')
    ax.patch.set_visible(False)
    for spine_name in ('left', 'right', 'top', 'bottom'):
        ax.spines[spine_name].set_visible(False)
    ax.spines[side].set_visible(True)
    if outward:
        ax.spines[side].set_position(('outward', float(outward)))
    ax.yaxis.set_label_position(side)
    ax.yaxis.set_ticks_position(side)
    ax.tick_params(
        axis='y',
        left=side == 'left',
        labelleft=side == 'left',
        right=side == 'right',
        labelright=side == 'right',
        labelsize=8.0,
        pad=2.0,
    )
    ax.tick_params(axis='x', bottom=False, top=False, labelbottom=False, labeltop=False)
    ax.set_ylabel(label, fontsize=9.0, labelpad=2.0)
    if limits is not None:
        ax.set_ylim(limits)


def tess_panel_column_spans(count: int, *, total_windows: int):
    if count == 1:
        return [(0, 6)]
    if total_windows <= 2:
        return [(0, 3), (3, 6)]
    if count == 2:
        return [(1, 3), (3, 5)]
    return [(0, 2), (2, 4), (4, 6)]


def render_time_domain_with_tess_panels(candidate_id: str):
    spec, source_names = build_time_domain_spec(candidate_id)
    raw_panel = next(panel for panel in spec.panels if panel.panel_id == 'raw')
    raw_traces = [
        trace for trace in spec.traces if trace.panel_id == 'raw' and trace.kind != 'line'
    ]
    optical_traces = [trace for trace in raw_traces if trace_group(trace) == 'optical']
    infrared_traces = [trace for trace in raw_traces if trace_group(trace) == 'infrared']
    ztf_traces = [trace for trace in raw_traces if trace_group(trace) == 'ztf']
    ps1_traces = [trace for trace in raw_traces if trace_group(trace) == 'ps1']
    tess_windows = split_tess_windows(spec) if SHOW_TESS_PANELS else []

    n_tess_rows = (
        int(np.ceil(len(tess_windows) / TESS_PANELS_PER_ROW))
        if tess_windows
        else 0
    )
    figure_height = FIGSIZE[1] + TESS_ROW_HEIGHT * n_tess_rows
    left_margin = (
        MAIN_MARGIN_WITH_OUTER_AXIS
        if ztf_traces
        else MAIN_MARGIN_WITHOUT_OUTER_AXIS
    )
    right_margin = 1.0 - (
        MAIN_MARGIN_WITH_OUTER_AXIS
        if ps1_traces
        else MAIN_MARGIN_WITHOUT_OUTER_AXIS
    )

    with plt.rc_context(PUBLICATION_STYLE):
        fig = plt.figure(figsize=(FIGSIZE[0], figure_height))
        fig.patch.set_facecolor('white')
        fig.patch.set_alpha(1.0)

        if tess_windows:
            outer = fig.add_gridspec(
                4,
                1,
                height_ratios=(4.0, 0.22, 1.35 * n_tess_rows, 0.68),
                hspace=0.30,
                left=left_margin,
                right=right_margin,
                bottom=0.045,
                top=0.955,
            )
            optical_ax = fig.add_subplot(outer[0])
            timeline_ax = fig.add_subplot(outer[1], sharex=optical_ax)
            tess_grid = outer[2].subgridspec(
                n_tess_rows,
                6,
                hspace=0.66,
                wspace=0.55,
            )
            legend_ax = fig.add_subplot(outer[3])
        else:
            outer = fig.add_gridspec(
                2,
                1,
                height_ratios=(4.0, 0.68),
                hspace=0.24,
                left=left_margin,
                right=right_margin,
                bottom=0.055,
                top=0.94,
            )
            optical_ax = fig.add_subplot(outer[0])
            timeline_ax = None
            tess_grid = None
            legend_ax = fig.add_subplot(outer[1])

        optical_ax.set_facecolor('white')
        infrared_ax = optical_ax.twinx() if infrared_traces else None
        ztf_ax = optical_ax.twinx() if ztf_traces else None
        ps1_ax = optical_ax.twinx() if ps1_traces else None
        axis_by_group = {
            'optical': optical_ax,
            'infrared': infrared_ax or optical_ax,
            'ztf': ztf_ax or optical_ax,
            'ps1': ps1_ax or optical_ax,
        }

        legend_handles = []
        legend_labels = []
        seen_labels = set()
        for trace in raw_traces:
            group = trace_group(trace)
            if group == 'tess':
                continue
            target_ax = axis_by_group[group]
            if group == 'infrared':
                alpha, marker_size = 0.88, 3.0
            elif group in {'ztf', 'ps1'}:
                alpha, marker_size = 0.80, 2.4
            else:
                alpha, marker_size = 0.78, 2.3
            label = str(trace.label or '') if trace.showlegend else '_nolegend_'
            handle = plot_standard_trace(
                target_ax,
                trace,
                alpha=alpha,
                marker_size=marker_size,
                label=label,
            )
            if handle is not None and label != '_nolegend_' and label not in seen_labels:
                legend_handles.append(handle)
                legend_labels.append(label)
                seen_labels.add(label)

        _style_lightcurve_axis(optical_ax)
        optical_ax.grid(False, which='both')
        optical_ax.set_xlabel(_axis_label_for_offset(spec.jd_offset))
        optical_ax.tick_params(axis='y', labelsize=8.0, pad=2.0)
        optical_ax.set_ylabel('ASAS-SN [mag]', fontsize=9.0, labelpad=2.0)
        if raw_panel.x_range is not None:
            optical_ax.set_xlim(raw_panel.x_range)
        optical_scale_traces = [
            trace for trace in optical_traces if str(trace.label or '') in {'g', 'V'}
        ] or optical_traces
        optical_limits = robust_magnitude_limits(optical_scale_traces)
        if optical_limits is not None:
            faint_limit, bright_limit = optical_limits
            faint_limit = np.ceil(faint_limit * 10.0) / 10.0
            bright_limit = np.floor(bright_limit * 10.0) / 10.0
            if faint_limit <= bright_limit:
                faint_limit = bright_limit + 0.1
            optical_ax.set_ylim(faint_limit, bright_limit)
        optical_ax.yaxis.set_major_locator(MultipleLocator(0.1))
        optical_ax.yaxis.set_major_formatter(FormatStrFormatter('%.1f'))

        if infrared_ax is not None:
            infrared_scale_traces = [
                trace
                for trace in infrared_traces
                if str(trace.label or '') in {'NEOWISE W1', 'NEOWISE W2'}
            ]
            if not infrared_scale_traces:
                infrared_scale_traces = [
                    trace
                    for trace in infrared_traces
                    if str(trace.label or '') in {'AllWISE W1', 'AllWISE W2'}
                ] or infrared_traces
            configure_overlay_magnitude_axis(
                infrared_ax,
                side='right',
                outward=0.0,
                label='WISE [mag]',
                limits=robust_magnitude_limits(infrared_scale_traces),
            )

        if ztf_ax is not None:
            configure_overlay_magnitude_axis(
                ztf_ax,
                side='left',
                outward=OUTER_AXIS_OFFSET_POINTS,
                label='ZTF [mag]',
                limits=robust_magnitude_limits(ztf_traces),
            )

        if ps1_ax is not None:
            configure_overlay_magnitude_axis(
                ps1_ax,
                side='right',
                outward=OUTER_AXIS_OFFSET_POINTS,
                label='PS1 [mag]',
                limits=robust_magnitude_limits(ps1_traces),
            )

        window_colors = [
            plt.get_cmap('tab10')(index % 10)
            for index in range(len(tess_windows))
        ]
        if tess_windows and timeline_ax is not None and tess_grid is not None:
            timeline_ax.set_ylim(0.0, 1.0)
            timeline_ax.axhline(0.50, color='0.55', linewidth=0.55, zorder=0)
            timeline_ax.text(
                -0.012,
                0.50,
                'TESS coverage',
                transform=timeline_ax.transAxes,
                ha='right',
                va='center',
                fontsize=7.0,
            )

            for window_index, (window, color) in enumerate(
                zip(tess_windows, window_colors),
                start=1,
            ):
                window_x = np.asarray(window.x, dtype=float)
                x_lo = float(np.nanmin(window_x))
                x_hi = float(np.nanmax(window_x))
                x_center = 0.5 * (x_lo + x_hi)
                optical_ax.axvspan(
                    x_lo,
                    x_hi,
                    facecolor=color,
                    edgecolor='none',
                    alpha=0.075,
                    zorder=0,
                )
                timeline_ax.axvspan(
                    x_lo,
                    x_hi,
                    ymin=0.12,
                    ymax=0.88,
                    facecolor=color,
                    edgecolor='0.20',
                    linewidth=0.35,
                    alpha=0.78,
                    zorder=2,
                )
                timeline_ax.text(
                    x_center,
                    0.94,
                    f'T{window_index}',
                    ha='center',
                    va='bottom',
                    rotation=45,
                    rotation_mode='anchor',
                    fontsize=6.2,
                    color=color,
                    clip_on=False,
                )
            timeline_ax.set_axis_off()

            for row_index in range(n_tess_rows):
                row_start = row_index * TESS_PANELS_PER_ROW
                row_windows = tess_windows[row_start : row_start + TESS_PANELS_PER_ROW]
                column_spans = tess_panel_column_spans(
                    len(row_windows),
                    total_windows=len(tess_windows),
                )
                for panel_position, (window, column_span) in enumerate(
                    zip(row_windows, column_spans)
                ):
                    window_index = row_start + panel_position + 1
                    color = window_colors[window_index - 1]
                    tess_ax = fig.add_subplot(
                        tess_grid[row_index, column_span[0] : column_span[1]]
                    )

                    window_x = np.asarray(window.x, dtype=float)
                    window_y = np.asarray(window.y, dtype=float)
                    relative_y = window_y - float(np.nanmedian(window_y))
                    panel_trace = replace(
                        window,
                        y=relative_y,
                        label=f'T{window_index}',
                        showlegend=False,
                    )
                    plot_standard_trace(
                        tess_ax,
                        panel_trace,
                        alpha=0.76,
                        marker_size=1.15,
                        label='_nolegend_',
                    )

                    x_lo = float(np.nanmin(window_x))
                    x_hi = float(np.nanmax(window_x))
                    x_pad = max(0.04, 0.025 * (x_hi - x_lo))
                    tess_ax.set_xlim(x_lo - x_pad, x_hi + x_pad)
                    tess_limits = robust_magnitude_limits([panel_trace])
                    if tess_limits is not None:
                        tess_ax.set_ylim(tess_limits)

                    _style_lightcurve_axis(tess_ax)
                    tess_ax.grid(False, which='both')
                    tess_ax.set_facecolor('white')
                    tess_ax.set_title(
                        f'T{window_index}: {x_lo:.1f}--{x_hi:.1f} d',
                        loc='left',
                        fontsize=7.8,
                        fontweight='semibold',
                        color=color,
                        pad=2.0,
                    )
                    tess_ax.set_xlabel(
                        _axis_label_for_offset(spec.jd_offset),
                        fontsize=7.0,
                        labelpad=1.0,
                    )
                    if panel_position == 0:
                        tess_ax.set_ylabel(
                            'TESS [mag]',
                            fontsize=7.5,
                            labelpad=2.0,
                        )
                    else:
                        tess_ax.set_ylabel('')
                    tess_ax.tick_params(labelsize=6.8)
                    tess_ax.locator_params(axis='x', nbins=4)
                    tess_ax.locator_params(axis='y', nbins=4)
                    tess_ax.spines['top'].set_color(color)
                    tess_ax.spines['top'].set_linewidth(1.25)

        legend_ax.set_axis_off()
        if legend_handles:
            legend_ax.legend(
                legend_handles,
                legend_labels,
                loc='center',
                frameon=False,
                fontsize=6.7,
                ncol=min(4, len(legend_labels)),
                columnspacing=1.0,
                handletextpad=0.35,
            )

        _draw_header_boxes(
            optical_ax,
            left=spec.header_left or str(candidate_id).removeprefix('stv_'),
            right=spec.header_right,
        )

    return fig, spec, source_names


def fetch_preferred_spectrum(candidate_id: str):
    candidate_id = str(candidate_id)
    attempts = []
    candidate_rows = spectrum_rows.loc[
        spectrum_rows['candidate_id'].astype(str).eq(candidate_id)
    ]
    for survey in SPECTRUM_SOURCE_PRIORITY.get(candidate_id, ()):
        survey_rows = candidate_rows.loc[candidate_rows['survey'].astype(str).eq(survey)]
        for _, source_row in survey_rows.iterrows():
            source_row = source_row.copy()
            for key, value in SPECTRUM_ROW_OVERRIDES.get((candidate_id, survey), {}).items():
                source_row[key] = value
            result = fetch_spectrum(
                source_row,
                survey_key=survey,
                cache_dir=SPECTRUM_CACHE_DIR,
            )
            attempts.append((survey, result.status.value, result.message))
            if result.status == FetchStatus.OK and result.data is not None:
                redshift = pd.to_numeric(source_row.get('spectrum_redshift'), errors='coerce')
                redshift = float(redshift) if np.isfinite(redshift) else None
                return survey, result.data, redshift, attempts
    return None, None, None, attempts


def display_candidate_spectrum(candidate_id: str):
    candidate_id = str(candidate_id)
    if candidate_id not in SPECTRUM_SOURCE_PRIORITY:
        return None
    survey, spectrum_data, redshift, attempts = fetch_preferred_spectrum(candidate_id)
    if spectrum_data is None:
        attempt_text = '; '.join(
            f'{source}: {status}{f" ({message})" if message else ""}'
            for source, status, message in attempts
        ) or 'no matching spectrum rows'
        print(f'{candidate_id} | spectrum unavailable | {attempt_text}')
        return None

    survey_label = SPECTRUM_SOURCE_LABELS.get(survey, survey)
    with warnings.catch_warnings():
        warnings.filterwarnings('ignore', message='invalid value encountered in divide')
        spectrum_png = render_spectrum_png(
            spectrum_data,
            survey=survey_label,
            candidate_id=candidate_id,
            redshift=redshift,
            title=f'{survey_label} spectrum',
            max_line_fits=50,
        )
    print(f'{candidate_id} | spectrum={survey_label} | status=ok')
    display(NotebookImage(data=spectrum_png))
    return survey


def display_candidate(candidate_id: str):
    figure, spec, source_names = render_time_domain_with_tess_panels(candidate_id)
    source_labels = ['ASAS-SN', *[external_source_label(source) for source in source_names]]
    n_tess_windows = len(split_tess_windows(spec)) if SHOW_TESS_PANELS else 0
    print(
        f'{candidate_id} | status={spec.status} | '
        f'TESS windows={n_tess_windows} | sources={", ".join(source_labels)}'
    )
    for warning in spec.warnings:
        if 'flux plotted as relative magnitude' in warning:
            continue
        print(f'  Warning: {warning}')
    display(figure)
    plt.close(figure)
    display_candidate_spectrum(candidate_id)
    return spec

## Plot one selected candidate

Change `CANDIDATE_INDEX` in the configuration cell or replace `selected_candidate_id` with a candidate ID from the cohort table.

In [ ]:
if cohort.empty:
    raise RuntimeError('No marked dippers with NEOWISE photometry were found.')
if not 0 <= CANDIDATE_INDEX < len(cohort):
    raise IndexError(f'CANDIDATE_INDEX must be between 0 and {len(cohort) - 1}.')

selected_candidate_id = str(cohort.iloc[CANDIDATE_INDEX]['candidate_id'])
selected_spec = display_candidate(selected_candidate_id)

## Plot the rest of the cohort

With `PLOT_ALL = True`, this produces one combined all-survey figure for every remaining candidate. Every contiguous TESS observing window receives its own chronological panel beneath the shared time-domain plot. The selected candidate above is skipped here so each object appears once.

In [ ]:
if PLOT_ALL:
    for candidate_id in cohort['candidate_id'].astype(str):
        if candidate_id == selected_candidate_id:
            continue
        display_candidate(candidate_id)
else:
    print('PLOT_ALL is False; only the selected candidate was rendered.')